In [ ]:
# Install required packages
# pip install langchain-text-splitters nltk sentence-transformers tiktoken

import re
from typing import List, Dict
import nltk
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data for sentence splitting
nltk.download('punkt')

# Sample text for demonstration
sample_text = """
Artificial Intelligence (AI) is transforming how we interact with technology. 
Machine learning, a subset of AI, enables computers to learn from data without explicit programming.

Deep learning uses neural networks with multiple layers. These networks can learn hierarchical representations of data. Natural Language Processing (NLP) allows computers to understand human language.

There are several types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Each has different applications and use cases.

Supervised learning uses labeled data to train models. The algorithm learns to map inputs to outputs based on example input-output pairs.
Unsupervised learning finds patterns in unlabeled data. Common techniques include clustering and dimensionality reduction.
Reinforcement learning trains agents through rewards and punishments in an environment.
"""

In [ ]:
s = """Artificial Intelligence (AI) is transforming how we interact with technology. 
Machine learning, a subset of AI, enables computers to learn from data."""

In [ ]:
len(s)

In [ ]:
def naive_chunking(text: str, chunk_size: int = 100, chunk_overlap: int = 20) -> List[str]:
    """
    Simple fixed-size chunking with overlap
    """
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        # Move forward by chunk_size minus overlap
        start += chunk_size - chunk_overlap
        
        # Stop if we're not making progress
        if start >= text_length:
            break
            
    return chunks

# Test naive chunking
naive_chunks = naive_chunking(sample_text, chunk_size=150, chunk_overlap=30)
print("=== Naive Chunking ===")
for i, chunk in enumerate(naive_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}...")

In [ ]:
def sentence_chunking(text: str, sentences_per_chunk: int = 3) -> List[str]:
    """
    Chunk by grouping fixed number of sentences
    """
    # Split into sentences using NLTK
    sentences = nltk.sent_tokenize(text)
    print(f"sentences : {sentences}")
    
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    
    return chunks

# Test sentence chunking
sentence_chunks = sentence_chunking(sample_text, sentences_per_chunk=2)
print("\n=== Sentence Chunking ===")
for i, chunk in enumerate(sentence_chunks):
    print(f"Chunk {i+1}: {chunk}")

In [ ]:
class SemanticSplitter:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
    
    def semantic_chunking(self, text: str, similarity_threshold: float = 0.5) -> List[str]:
        """
        Split text based on semantic similarity between sentences
        """
        # Split into sentences
        sentences = nltk.sent_tokenize(text)
        print(f"sentences : {sentences}")
        print(f"len(sentences) :{len(sentences)}")
        
        if len(sentences) <= 1:
            return [text]
        
        # Get sentence embeddings
        embeddings = self.model.encode(sentences)

        print(f"embeddings : {embeddings}")
        
        # Calculate similarities between consecutive sentences
        chunks = []
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            print(f" count : {i}")
            # Calculate cosine similarity between current and previous sentence
            similarity = cosine_similarity(
                [embeddings[i-1]], 
                [embeddings[i]]
            )[0][0]

            print(f"similarity : {similarity}")
            
            if similarity >= similarity_threshold:
                # Similar enough, add to current chunk
                print(f"current_chunk in if  : {current_chunk}")
                current_chunk.append(sentences[i])
                print(f"current_chunk in if  after append : {current_chunk}")
            else:
                # Significant topic shift, start new chunk
                print(f"chunks in else before join {chunks}")
                chunks.append(' '.join(current_chunk))
                print(f"chunks in else after join {chunks}")
                print(f"current_chunk in else {current_chunk}")
                current_chunk = [sentences[i]]
                print(f"current_chunk = [sentences[i]] : {current_chunk} ")
        
        # Add the last chunk
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks

# Test semantic chunking
splitter = SemanticSplitter()
semantic_chunks = splitter.semantic_chunking(sample_text, similarity_threshold=0.3)
print("\n=== Semantic Chunking ===")
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i+1}: {chunk}")

In [ ]:
labels = [0, 0, 0, 1, 1, 2, 2]
buckets = {}
buckets2 = {}
for i, lab in enumerate(labels):
    # print(i,lab)
    # buckets.setdefault(int(lab), []).append(i)
    if lab in buckets2.keys():
        buckets2[lab].append(i)
    else:
        buckets2[lab] = [i]

print(buckets2)

In [ ]:
labels = [0, 0, 0, 1, 1, 2, 2,2,2,3,4,4,5,5,5]
buckets = {}
for i, lab in enumerate(labels):
    buckets.setdefault(lab, []).append(i)

print(buckets)

In [ ]:
for k,v in buckets.items():
    print(k,min(v))

In [ ]:
ordered_clusters = sorted(buckets.items(), key=lambda x: min(x[1]))
ordered_clusters